# DT401 Week 7 Section 1: Working Through a Data Analysis

**Module:** Data Fundamentals  
**Week:** 7 of 7  
**Section:** 1 of 2  
**Estimated Time:** 60 minutes  

---

## Learning Objectives

By the end of this notebook, you will be able to:
- Build multiple regression models to solve a business forecasting problem
- Identify and address multicollinearity in predictor variables
- Evaluate model performance using R², RMSE, and MAE
- Compare models to select the best approach for a given business problem
- Validate models using test-train split to detect overfitting

---

## Context Recap

Over the past six weeks, you have developed skills in data types, quality principles, data-gathering methods, and analysis techniques. Week 7 brings everything together in an integrated guided project — applying the complete analytical workflow to a real-world forecasting problem.

You will forecast outdoor event attendance from weather conditions, demonstrating the end-to-end process professionals use in workplace settings.

**Dataset:** Open-Meteo weather observations, London, UK (51.49°N, 0.16°W), January 2023–January 2026.

---

## How to Use This Notebook

This notebook follows a complete analytical workflow — from defining a business question through to a validated model ready to deploy. Each Part corresponds to one stage of that journey. As you run the code cells, read the explanations carefully: understanding *what the code is doing and why* matters more here than being able to write it yourself.

Throughout the notebook, coloured boxes signal different types of content: **yellow** boxes are reminders connecting the work to your broader learning; **blue** boxes explain what just happened in the code; **green** boxes are tasks for you to complete.


---

## Part 1: The Business Problem

Every analysis starts with a clear question. Before writing a single line of code, a data analyst defines what they are trying to predict, what counts as success, and how the answer will be used. This Part establishes that foundation for everything that follows.

### Scenario

An outdoor events company hosts weekly community festivals in London parks. Attendance varies significantly based on weather conditions — sunny days attract hundreds of visitors while rainy days see cancellations. The company wants to forecast attendance 24 hours ahead to optimise:

- **Staffing:** Right number of volunteers and security personnel
- **Catering:** Food and beverage quantities (reducing waste and costs)
- **Customer Communication:** Early warnings about likely conditions

**Business Question:** Can we predict event attendance from weather forecasts?

**Success Criteria:** Forecast within ±10% of actual attendance, with the ability to identify the weather variables that most strongly influence attendance.

### Similar Challenges Across Sectors

| Sector | Forecasting Challenge | Weather Link |
|:-------|:---------------------|:-------------|
| Healthcare | Predict A&E admissions during heatwaves and cold snaps | Temperature, humidity |
| Retail | Forecast footfall and sales | Temperature, precipitation |
| Local Government | Anticipate gritting and emergency service demand | Snowfall, ice risk |
| Charity | Plan outdoor fundraising events | All weather variables |

### The Analytical Workflow

This notebook follows a standard analytical workflow used by data analysts across sectors. The diagram below shows each stage — we will work through them in order, and you can refer back to it at any point to orientate yourself within the overall process.


```mermaid
flowchart LR
    A[Define Business Question] --> B[Explore & Prepare Data]
    B --> C[Build Model 1: All Variables]
    C --> D[Build Model 2: Simplified]
    D --> E[Validate on Unseen Data]
    E --> F[Interpret for Stakeholders]

    style A fill:#ffffff,stroke:#000000,color:#000000,stroke-width:2px
    style B fill:#ffffff,stroke:#000000,color:#000000,stroke-width:2px
    style C fill:#ffffff,stroke:#000000,color:#000000,stroke-width:2px
    style D fill:#ffffff,stroke:#000000,color:#000000,stroke-width:2px
    style E fill:#ffffff,stroke:#000000,color:#000000,stroke-width:2px
    style F fill:#ffffff,stroke:#000000,color:#000000,stroke-width:2px
```


Each Part of this notebook corresponds to one stage in the workflow above. We are currently at stage one — defining the business question. By the time you reach the Summary, you will have completed every stage, producing a validated, deployable model. We will refer back to this diagram at each transition so you always know where you are in the process.


<div class="alert alert-block alert-warning">
<b>Reminder:</b><br>
This guided project demonstrates techniques directly applicable to your workplace. As you work through each section, consider similar forecasting challenges in your own organisation — demand planning, resource allocation, and service capacity decisions all benefit from the same analytical approach.
</div>

---

## Part 2: Data Exploration

With the business question established, the first analytical step is always to load and understand the data. The code cells below handle data loading and preparation — read the explanations to follow what each step does, then run the cells to see the results. You do not need to write or modify any code in this Part; focus on understanding what is happening and why.


Raw data rarely arrives in a clean, ready-to-use state. The diagram below shows the preparation steps this notebook follows before any analysis can begin.


```mermaid
flowchart TD
    A[Raw CSV File: Hourly weather observations] --> B[Load & Clean Column Names]
    B --> C[Filter Valid Date Rows]
    C --> D[Convert Text to Numeric Values]
    D --> E[Select Key Variables]
    E --> F[Simulate Attendance from Weather]
    F --> G[Clean Dataset Ready for Analysis]

    style A fill:#ffffff,stroke:#000000,color:#000000,stroke-width:2px
    style B fill:#ffffff,stroke:#000000,color:#000000,stroke-width:1px
    style C fill:#ffffff,stroke:#000000,color:#000000,stroke-width:1px
    style D fill:#ffffff,stroke:#000000,color:#000000,stroke-width:1px
    style E fill:#ffffff,stroke:#000000,color:#000000,stroke-width:1px
    style F fill:#ffffff,stroke:#000000,color:#000000,stroke-width:1px
    style G fill:#ffffff,stroke:#000000,color:#000000,stroke-width:3px
```

The data arrives as a raw CSV file with messy column headers and mixed data types — both common data quality issues you encountered in earlier weeks. The code below cleans and prepares it step by step. You do not need to write this code; focus on understanding what each step achieves and why it is necessary before modelling can begin.

### Importing Libraries and Setting Up the Environment

Before we load the data, we need to import the necessary libraries and set up our environment. This will help you to check that you have the right tools available before we dive into the data preparation steps.

In [ ]:
# Import required libraries
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split

# Suppress matplotlib backend warnings in notebooks
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')

print("Libraries imported successfully. Ready to load and prepare the data.")

### Loading and Cleaning the Data

Now we are ready to load the data and clean it. The code below handles the messy column names and mixed data types, ensuring that the dataset is in a usable format for analysis.

The data comes from Open-Meteo, a free weather API, and contains hourly observations for London covering January 2023 to January 2026. 

The cleaning steps handle two specific issues: 

- the CSV file has inconsistent column names (with units like `(°C)` embedded in them)
-  some numeric columns arrive as text rather than numbers

Both of these would cause errors if left untreated.


In [ ]:
# Load weather dataset
weather_data = pd.read_csv('../open-meteo-51.49N0.16W23m.csv', skiprows=3)  # Skip the first 3 metadata rows in the CSV
weather_data.columns = weather_data.columns.str.split(' ').str[0].str.strip()  # Clean column names: remove units like '(°C)' and extra spaces
weather_data = weather_data.rename(columns={'time': 'date'})
weather_data = weather_data[weather_data['date'].str.match(r'^\d{4}-\d{2}-\d{2}T', na=False)].reset_index(drop=True)  # Keep only rows with valid date formats

numeric_cols = weather_data.columns[1:]
weather_data[numeric_cols] = weather_data[numeric_cols].apply(pd.to_numeric, errors='coerce')  # Convert text to numbers; unconvertible values become NaN (missing)

print(f"Dataset loaded: {len(weather_data):,} hourly observations")

print(f"Available columns: {list(weather_data.columns[:8])}")
weather_data[['date', 'temperature_2m', 'precipitation', 'wind_speed_10m', 'cloud_cover', 'sunshine_duration']].head()

### Simulating Attendance Data

Since we do not have real attendance records, the next cell creates simulated attendance data based on the weather. 

The simulation uses a formula where temperature increases attendance, while rain, wind, and cloud cover decrease it — plus some random variation to reflect real-world unpredictability. In a real project, you would load actual attendance data from a ticketing system or entry counter instead.

In [ ]:
# Create simulated attendance data based on weather conditions
# In a real project you would load actual attendance records
np.random.seed(42)

# Select the key weather variables and simulate attendance: base + temperature effect - rain penalty - wind penalty - cloud cover penalty + sunshine duration effect + random noise
data = (weather_data[['date', 'temperature_2m', 'precipitation', 'wind_speed_10m', 'cloud_cover', 'sunshine_duration']]
    .copy()
    .dropna()
    .assign(attendance=lambda df: (
        200 +                                             # Base attendance
        (df['temperature_2m'] * 8) +                      # Warmer weather increases attendance
        (df['precipitation'] * -220) +                    # Rain significantly reduces attendance
        (df['wind_speed_10m'] * -3) +                     # Wind deters some attendees
        (df['cloud_cover'] * -0.4) +                      # Cloud cover reduces attendance
        (df['sunshine_duration'].apply(np.sqrt) * 0.01) + # More sunshine in an hour increases attendance
        np.random.normal(0, 30, len(df))                  # Random variation
    ).clip(0, 500).astype(int))
)

print(f"Data prepared: {len(data):,} observations")
print(f"Date range: {data['date'].min()} to {data['date'].max()}\n")
print(f"Variable ranges:")
for col in ['temperature_2m', 'precipitation', 'wind_speed_10m', 'cloud_cover', 'sunshine_duration', 'attendance']:
    print(f"  {col:25}: {data[col].min():.1f} to {data[col].max():.1f}")
data.head()

You can see that the data table now has a new `attendance` column, which is the target variable we will try to predict from the weather conditions. 

### Descriptive Statistics

To understand the data better, we can look at summary statistics. This gives us insights into the typical values and variability of each variable, which can inform our modelling decisions later on. 

In [ ]:
data.describe().round(2)

From this we can also see that there are no missing values in the dataset, as the count of non-null values matches the total number of rows for each variable. This is important to know before we start building models, as missing data can cause errors or bias in our results if not handled properly.

We can also see that the maximum and minimum values for each variable are within expected ranges, which gives us confidence that the data is reasonable and does not contain obvious errors or outliers. For example, temperatures range from -5°C to 34°C, which is plausible for London weather.

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

- We loaded hourly weather data covering approximately 3 years (2023–2026)
- We created a simulated attendance variable incorporating temperature, rain, wind, and cloud cover
- Attendance per hour ranges from 0 to 500 people (representing realistic variation for community festivals)
- The dataset has no missing values after filtering

**Key lesson:** Always start analysis by understanding data characteristics — ranges, distributions, and any missing values — before building models.
</div>

### Visualising the Attendance Distribution

To understand the distribution of our target variable (attendance), we can create a histogram. This will show us how attendance values are spread across the dataset, and whether there are any patterns or outliers that we should be aware of before modelling.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4), constrained_layout=True)
ax.hist(data['attendance'], bins=30, edgecolor='black', alpha=0.7, color='steelblue')
ax.set_xlabel('Attendance (people)')
ax.set_ylabel('Frequency (number of days)')
ax.set_title('Distribution of Event Attendance per Hour')
ax.axvline(data['attendance'].mean(), color='red', linestyle='--',
           label=f'Mean: {data["attendance"].mean():.0f} people')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

print(f"Attendance Statistics:")
print(f"  Mean:    {data['attendance'].mean():.0f} people")
print(f"  Std Dev: {data['attendance'].std():.0f} people")
print(f"  Range:   {data['attendance'].min()} to {data['attendance'].max()} people")

**Conclusion:** Attendance is roughly normally distributed around 230 people with a standard deviation of approximately ±75 people. This variability creates operational challenges, such as unpredictable demand for staffing, catering, and resources. Accurate forecasts reduce waste and improve service quality, translating directly into cost savings and customer satisfaction.

### Identifying Relationships: Correlation Matrix

To understand how the weather variables relate to each other and to attendance, we can calculate a correlation matrix. This will show us the strength and direction of linear relationships between all pairs of variables.

In [ ]:
# Correlation matrix across all variables including attendance
variables = ['temperature_2m', 'precipitation', 'wind_speed_10m', 'cloud_cover', 'sunshine_duration', 'attendance']
correlation_matrix = data[variables].corr()

# Visualise as heatmap
fig, ax = plt.subplots(figsize=(8, 6), constrained_layout=True)
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f',
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Correlation Matrix: Weather Variables and Attendance')
plt.show()

# Show correlations with attendance specifically
print("\nCorrelations with Attendance (sorted by strength):")
print("-" * 55)
attendance_corr = correlation_matrix['attendance'].drop('attendance').sort_values(key=abs, ascending=False)
for var, corr in attendance_corr.items():
    strength = 'Strong' if abs(corr) > 0.5 else 'Moderate' if abs(corr) > 0.3 else 'Weak'
    direction = 'positive' if corr > 0 else 'negative'
    print(f"  {var:25} | {corr:+.3f} | {strength} {direction}")

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

The correlation matrix shows how strongly each weather variable relates to attendance:

| Variable | Correlation | Interpretation |
|:---------|:-----------|:---------------|
| Temperature | +0.61 | Strong positive. Warmer weather increases attendance |
| Precipitation | −0.53 | Moderate negative. Rain reduces attendance |
| Sunshine Duration | +0.37 | Moderate positive. More sunshine increases attendance |
| Cloud Cover | −0.36 | Moderate negative. People prefer going out in clear skies |
| Wind Speed | −0.23 | Weak negative. Wind deters some attendees |

**Key lesson:** Correlation matrix helps identify promising predictor variables before building models. Temperature and precipitation are the strongest candidates.
</div>

<div class="alert alert-block alert-warning">
<b>Predicting people is challenging!</b><br>
While we have identified some strong relationships between weather and attendance, in real-world scenarios, many other factors can influence attendance, such as holidays, day of week, special events, and even local construction. These factors can create noise in the data, making it harder to predict attendance accurately. 

For this reason, it is rare to find strong correlations above ±0.3 when predicting human behaviour. Teasing out meaningful patterns from noisy data is a key skill for data analysts, and it requires careful model building and validation to ensure that we are capturing real relationships rather than random fluctuations.
</div>

### Multicollinearity Check

Notice an important pattern in the correlation matrix: **temperature and sunshine duration are positively correlated (around 0.43)**. This makes intuitive sense, as sunshine warms the ground and air. This is called **multicollinearity**: when two predictor variables are strongly correlated with each other.

**Why does this matter?**

When we want to build a model, we want to include as much information as possible to make accurate predictions. We want to include as many relevant predictor variables as possible to capture the complexity of the relationships in the data. 

However, each additonal variable also adds noise and complexity to the model. Both noise and complexity can reduce the model's ability to generalise to new data, which is ultimately what we want from a predictive model. 

When two predictor variables are highly correlated, they provide overlapping information about the dependent variable. It can be hard to determine which one is actually driving the relationship with the dependent variable. Is sunshine or temperature the primary driver of variation in attendance? 

Removing one of the correlated variables can simplify the model and improve its interpretability without sacrificing much predictive power.

<div class="alert alert-block alert-info">
<b>Rule of Thumb</b><br>
If two predictor variables correlate more strongly with each other than either does with the variable we are trying to predict (the dependent variable), consider removing one.
</div>

Let us have a look at the correlations between the predictor variables, also known as the independent variables, to see if there are any strong correlations that might indicate multicollinearity.

In [ ]:
# Check multicollinearity between predictor variables
predictor_vars = ['temperature_2m', 'precipitation', 'wind_speed_10m', 'cloud_cover', 'sunshine_duration']
predictor_corr = data[predictor_vars].corr()

fig, ax = plt.subplots(figsize=(7, 5), constrained_layout=True)
sns.heatmap(predictor_corr, annot=True, cmap='coolwarm', center=0, fmt='.2f',
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Predictor Variable Correlations (Multicollinearity Check)')
plt.show()

**Conclusion:** Temperature and sunshine duration show a moderate positive correlation (0.43), which suggests that they might be providing overlapping information about attendance. Sunshine duration is also a weaker predictor of attendance than it is correlated with temperature (0.37 vs 0.43) and temperature correlates more strongly with attendance than sunshine duration (0.61 vs 0.43), which suggests that temperature might be the more important variable to keep in the model if we want to simplify it later on.

We might consider removing sunshine duration when we simplify the model in Part 4.

Now that we understand our data and have identified which variables relate to attendance, we can build our first predictive model. Looking back at the workflow diagram from Part 1, we are moving from **Explore & Prepare Data** into **Build Model 1: All Variables**.

---

## Part 3: Building Model 1 — All Variables

Exploration showed that temperature, precipitation, wind speed, and cloud cover all correlate with attendance. The natural starting point is a model that uses all four variables — this gives us a performance baseline to compare against simpler alternatives. Think of this as establishing a ceiling: if removing a variable does not significantly reduce performance, the simpler model is likely preferable.


### Building the Linear Regression Model

The next cell uses **scikit-learn** (imported as `sklearn`), a widely-used Python library for building and evaluating models. 

The model `LinearRegression` fits a straight-line relationship between the weather variables and attendance — the same kind of linear regression you explored in earlier weeks, now applied with multiple predictor variables simultaneously.

In [ ]:
# Define independent (X) and dependent (y) variables — all 4 weather predictors
X_model1 = data[['temperature_2m', 'precipitation', 'wind_speed_10m', 'cloud_cover', 'sunshine_duration']]
y = data['attendance']

# Build and fit the model
model_1 = LinearRegression()
model_1.fit(X_model1, y)

# Display coefficients
print("\nCoefficients:")
for var, coef in zip(X_model1.columns, model_1.coef_):
    print(f"  {var:25} | {coef:+7.3f}")
print(f"  Intercept {' ':15} | {model_1.intercept_:+7.2f}")

**Coefficient Interpretation:**

Each coefficient tells us the expected change in attendance for a one-unit increase in that variable, holding all other variables constant:

- **Temperature (+):** Each 1°C increase adds approximately 8 attendees — warmer days draw larger crowds
- **Precipitation (−):** Each 1mm rain reduces attendance by approximately 96 people — rain is the strongest deterrent
- **Wind Speed (−):** Each 1 km/h increase reduces attendance by approximately 3 people — wind has a smaller effect
- **Cloud Cover (−):** Each 10% increase in cloud cover reduces attendance by approximately 5 people
- **Sunshine Duration (+):** Each full hour of sunshine increases attendance by approximately 3.6 people, as there are 3600 seconds in an hour and the sunshine duration variable is measured in seconds, so we need to convert it to hours to interpret the coefficient in a meaningful way. 
- **Intercept:** The baseline attendance when all weather variables are zero (0°C, no rain, no wind, clear skies). In this case it is around 204 people.

**Practical implications:** A sunny 25°C clear day versus a 15°C overcast day with light rain can differ by 150+ attendees. This quantitative foundation enables concrete operational planning.


<div class="alert alert-block alert-info">
<b>Further reading</b><br>

[Scikit-learn: LinearRegression documentation (Scikit-learn Development Team, 2024)](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html) – Reference page for the model used in this notebook, including parameter explanations and worked examples.
</div>

#### Evaluating Model Accuracy

To evaluate how well our model predicts attendance, we can calculate several performance metrics:

- **R²**: The proportion of variance in attendance explained by the model. Ranges from 0 to 1, with higher values indicating better fit.
- **RMSE (Root Mean Squared Error)**: The average magnitude of prediction errors. The lower the RMSE, the better the model's predictions match the actual attendance values. Equally, the RMSE can be  thought of as a measure of prediction error / uncertainty, simmilar to the standard deviation of the prediction errors.
- **MAE (Mean Absolute Error)**: The average absolute difference between predicted and actual attendance values. Like RMSE, lower is better, but MAE is less sensitive to outliers (unusual values) than RMSE.

Let's calculate these metrics for our model to understand its performance in business-relevant terms.

In [ ]:
# Calculate and display model performance metrics
predictions_1 = model_1.predict(X_model1)
r2_1 = r2_score(y, predictions_1)
rmse_1 = np.sqrt(mean_squared_error(y, predictions_1))
mae_1 = mean_absolute_error(y, predictions_1)

print(f"  R-squared:  {r2_1:.3f}  ({r2_1*100:.1f}% of attendance variation explained)")
print(f"  RMSE:       {rmse_1:.1f} attendees  (typical forecast error)")
print(f"  MAE:        {mae_1:.1f} attendees  (average absolute error)")
print(f"\n  For a 250-person event, expect: {250-rmse_1:.0f}–{250+rmse_1:.0f} actual attendance")

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

- **R² ≈ 0.77:** The model explains 77% of attendance variation. The remaining 23% reflects factors not captured by weather alone (marketing campaigns, competing events, public holidays).
- **RMSE ≈ 37 attendees:** Typical forecast error is ±37 people — for a 250-person event that is approximately 15% error.
- **MAE ≈ 27 attendees:** On average, the forecast is off by 27 people in either direction.

The R² tells us overall fit, while RMSE and MAE give us a concrete measure of how far off our predictions are from the actual attendance figures (the forecast uncertainty).
</div>

**Conclusion:** An RMSE of ±37 people represents approximately 15% error for a typical 250-person event. This is within the acceptable range for operational planning where staffing ratios tolerate 15–20% variation. However, we can try to improve this by refining the model.

---

## Part 4: Building Model 2 — Simplified (Addressing Multicollinearity)

In Part 2, we found that temperature and sunshine duration are correlated with each other (approximately 0.43) — this multicollinearity means including both adds complexity without proportional benefit. Let's test whether removing sunshine duration significantly reduces model performance. The model comparison table after this task summarises which approach to carry forward into validation.


<div class="alert alert-block alert-success">
<b>Your turn:</b><br>
Remove sunshine duration from the model and compare performance to Model 1. Does the simplified model perform similarly?

<details>
    <summary style="font-weight:bold">Click here for a hint</summary>

Steps:
1. Create `X_model2` using only `temperature_2m`, `precipitation`, `wind_speed_10m` and `cloud_cover`
2. Fit a new `LinearRegression()` model on `X_model2`
3. Calculate predictions, R², and RMSE
4. Compare to Model 1 results

</details>
</div>



<details>
    <summary style="color:green;font-weight:bold">Click here for the solution</summary>

```python
# Model 2: Remove sunshine_duration (correlated with temperature)
X_model2 = data[['temperature_2m', 'precipitation', 'wind_speed_10m', 'cloud_cover']]

model_2 = LinearRegression()
model_2.fit(X_model2, y)

predictions_2 = model_2.predict(X_model2)
r2_2 = r2_score(y, predictions_2)
rmse_2 = np.sqrt(mean_squared_error(y, predictions_2))
mae_2 = mean_absolute_error(y, predictions_2)

print(f"Model 2 R²:   {r2_2:.3f}")
print(f"Model 2 RMSE: {rmse_2:.1f}")
print(f"Model 2 MAE:  {mae_2:.1f}")
print(f"\nR² change from Model 1: {r2_2-r2_1:+.3f}")
print(f"RMSE change:             {rmse_2-rmse_1:+.1f}")
```
</details>

In [ ]:
# Your code here — build Model 2 without sunshine_duration



### Model Comparison

| Model | Variables | R² | RMSE | Notes |
|:------|:----------|:---|:-----|:------|
| Model 1 | temperature, precipitation, wind, cloud | ~0.77 | ~37 | All predictors; multicollinearity risk |
| Model 2 | temperature, precipitation, wind | ~0.77 | ~37 | Simpler; addresses multicollinearity |

**Business Decision:** Model 2 is preferred:
- **Easier to communicate:** Stakeholders immediately understand temperature, rain, wind and cloud cover. Sunshine duration adds complexity with out sufficient benefit
- **Operational simplicity:** Four data inputs needed for forecasting versus five
- **Reduced overfitting risk:** Simpler models generalise better to new, unseen data
- **Better interpretability:** Coefficients are more stable and easier to explain without the confounding effect of sunshine duration

**Key insight:** Analytical judgement involves knowing when to remove variables. A simpler model that is easier to use and explain is often preferable to one that is more accurate but harder to maintain.


The diagram below illustrates the decision logic: when two models perform similarly, the simpler one is usually the better choice.


```mermaid
flowchart TD
    A[Model 1: 4 Variables] --> B{Performance drop significant?}
    B -- "No: ~0.01 R² drop" --> C[Prefer Simpler Model]
    B -- "Yes: large drop" --> D[Keep Complex Model]
    C --> E[Model 2: 3 Variables]
    E --> F[Easier to explain to stakeholders]
    E --> G[Fewer data inputs nneeded]
    E --> H[Reduced overfitting risk]

    style A fill:#ffffff,stroke:#000000,color:#000000,stroke-width:2px
    style B fill:#ffffff,stroke:#000000,color:#000000,stroke-width:2px
    style C fill:#ffffff,stroke:#000000,color:#000000,stroke-width:3px
    style D fill:#ffffff,stroke:#000000,color:#000000,stroke-width:1px
    style E fill:#ffffff,stroke:#000000,color:#000000,stroke-width:2px
    style F fill:#ffffff,stroke:#000000,color:#000000,stroke-width:1px
    style G fill:#ffffff,stroke:#000000,color:#000000,stroke-width:1px
    style H fill:#ffffff,stroke:#000000,color:#000000,stroke-width:1px
```


When two models perform similarly, the simpler one is usually preferred. Model 2 loses very little accuracy but gains significant practical advantages — it is easier to explain, requires fewer data inputs, and is less likely to overfit. We will take Model 2 forward into validation in Part 5.


---

## Part 5: Model Validation with Test-Train Split

So far, we have evaluated both models on the same data used to build them. This is a problem: a model could perform well simply by memorising the training data, rather than learning genuine patterns. The test-train split addresses this by keeping a portion of the data hidden during model building, then evaluating performance on data the model has never seen.


The diagram below shows how the data is divided and why comparing training versus test performance is the key diagnostic step.


```mermaid
flowchart LR
    A[Full Dataset] --> B[80% Training Data]
    A --> C[20% Test Data]
    B --> D[Build Model]
    D --> E[Evaluate on Training Data]
    D --> F[Evaluate on Test Data]
    C --> F
    E --> G{Compare Performance}
    F --> G
    G -- "Similar" --> H[Model Generalises Well]
    G -- "Large Gap" --> I[Overfitting Risk]

    style A fill:#ffffff,stroke:#000000,color:#000000,stroke-width:2px
    style B fill:#ffffff,stroke:#000000,color:#000000,stroke-width:2px
    style C fill:#ffffff,stroke:#000000,color:#000000,stroke-width:2px
    style D fill:#ffffff,stroke:#000000,color:#000000,stroke-width:2px
    style E fill:#ffffff,stroke:#000000,color:#000000,stroke-width:1px
    style F fill:#ffffff,stroke:#000000,color:#000000,stroke-width:1px
    style G fill:#ffffff,stroke:#000000,color:#000000,stroke-width:2px
    style H fill:#ffffff,stroke:#000000,color:#000000,stroke-width:3px
    style I fill:#ffffff,stroke:#000000,color:#000000,stroke-width:1px
```


The key idea is simple: build the model on one portion of the data, then test it on data the model has never seen. If performance drops significantly on the test data, the model has learned noise rather than genuine patterns — it has **overfit**.

Think of overfitting like revising only past exam papers without understanding the underlying concepts. You might score perfectly on those specific papers, but struggle with new questions that test the same topics differently. A model that overfits has memorised its training data rather than learning the real patterns.

Let's apply this to our Model 2, which we identified as the better model in Part 4. We will split the data into training and test sets, build the model on the training data, and then evaluate its performance on both sets to check for overfitting.


In [ ]:
# Use Model 2 (simplified, no sunshine_duration) for validation
X_final = data[['temperature_2m', 'precipitation', 'wind_speed_10m', 'cloud_cover']]

# Split: 80% training, 20% testing — random_state ensures reproducibility
X_train, X_test, y_train, y_test = train_test_split(X_final, y, test_size=0.2, random_state=42)

print(f"Training set:  {len(X_train):,} observations  (80%)")
print(f"Test set:      {len(X_test):,} observations  (20%)")

In [ ]:
# Build model on training data ONLY
model_final = LinearRegression()
model_final.fit(X_train, y_train)

# Evaluate on training data
train_predictions = model_final.predict(X_train)
train_r2   = r2_score(y_train, train_predictions)
train_rmse = np.sqrt(mean_squared_error(y_train, train_predictions))

# Evaluate on test data (data the model has never seen)
test_predictions = model_final.predict(X_test)
test_r2   = r2_score(y_test, test_predictions)
test_rmse = np.sqrt(mean_squared_error(y_test, test_predictions))

print("=== Model Validation Results ===")
print(f"\n  Training R²:   {train_r2:.3f}")
print(f"  Training RMSE: {train_rmse:.1f} attendees")
print(f"\n  Test R²:       {test_r2:.3f}")
print(f"  Test RMSE:     {test_rmse:.1f} attendees")
print(f"\n  R² difference (train − test):   {train_r2 - test_r2:.3f}")
print(f"  RMSE difference (test − train): {test_rmse - train_rmse:+.1f} attendees")

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

- We split the data: 80% for training the model, 20% reserved for testing
- The model was built using ONLY training data
- We then evaluated it on both training and test data separately to compare performance.

**Small performance drop (train → test) indicates good generalisation.** A large drop would suggest overfitting — the model learned patterns specific to training data that don't hold in new data.

**Key lesson:** Always validate on unseen data. Models that look excellent on training data can fail badly in production if overfitting is not detected.
</div>

**Practical Interpretation:**

| Metric | Value | Conclusion |
|:-------|:------|:----------------|
| Training R² | ~0.76 | Model fits training data well |
| Test R² | ~0.78 | Model generalises to new data (minimal overfitting) |
| Test RMSE | ~36 | When forecasting NEW events, expect ±36 person accuracy |

**Deployment Decision:** Deploy for events where expected attendance exceeds 200. For those events, a ±36-person error (~18%) is operationally manageable for staffing and catering. For smaller events (<100 expected), percentage error is too high — augment with additional data sources (pre-sold tickets, social media engagement).


### Visualising Predictions vs Actual Attendance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)

# Left: scatter plot predictions vs actual
axes[0].scatter(y_test, test_predictions, alpha=0.4, color='steelblue', s=20)
perfect = [y_test.min(), y_test.max()]
axes[0].plot(perfect, perfect, 'r--', lw=2, label='Perfect predictions')
axes[0].set_xlabel('Actual Attendance')
axes[0].set_ylabel('Predicted Attendance')
axes[0].set_title('Predictions vs Actual (Test Data)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Right: residuals (errors) distribution
residuals = y_test.values - test_predictions
axes[1].hist(residuals, bins=30, edgecolor='black', alpha=0.7, color='coral')
axes[1].axvline(0, color='red', linestyle='--', lw=2, label='Zero error')
axes[1].set_xlabel('Forecast Error (Actual − Predicted)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Forecast Errors')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

fig.suptitle('Model Validation: How Good Are the Forecasts?', fontweight='bold')
plt.show()

print(f"Error Statistics:")
print(f"  Mean error (bias):   {residuals.mean():+.1f} people  (near zero = unbiased)")
print(f"  Std of errors:       {residuals.std():.1f} people")
print(f"  Max over-prediction: {residuals.min():.0f} people")
print(f"  Max under-prediction:{residuals.max():.0f} people")

**Business Insight:** Points clustering close to the red dashed line indicate accurate forecasts. Outliers (points far from the line) represent unusual events — extreme weather, competing festivals, or public holidays. In production, flag days where weather forecasts include extreme values for human review before relying on model output.

The residuals chart confirms errors are approximately normally distributed around zero (unbiased), which means the model is neither systematically over- nor under-predicting attendance.


Look at the residuals histogram (right panel). The errors are centred near zero — what does this tell you about the model's predictions? Now look at the scatter plot (left panel): are there any clusters of points far from the red dashed line? What might explain those outliers?

<details>
<summary>Solution</summary>

The residuals being centred near zero tells us the model is **unbiased** — it is not systematically over-predicting or under-predicting attendance. If the histogram were shifted noticeably to one side, that would indicate a consistent prediction error in one direction, which would be a serious problem for operational planning.

Points far from the red dashed line in the scatter plot represent days where the forecast was significantly wrong. These outliers likely correspond to unusual events — extreme weather conditions, competing local festivals, or public holidays — factors that a weather-only model cannot capture. In a production system, you would flag these cases for human review rather than acting on the model output alone.

</details>


---

## Part 6: Independent Task — Library Visitor Forecasting

You have now completed the full analytical workflow for the events scenario: defined the business question, explored the data, built and compared two models, and validated the preferred model on unseen data. This final task asks you to apply the same thinking to a new workplace problem.

This task assesses your analytical thinking, not your coding ability. You are designing a forecasting approach — the same kind of planning a data analyst does before writing any code. Aim for approximately 200 words across the template fields.

### Your Challenge

You work for a local council managing community libraries. Visitor numbers vary with weather conditions and day of week. Your manager asks: "Can we forecast library visitors to optimise staffing?"

**Design a complete forecasting approach:**

1. **Business Problem:** What are you predicting? What counts as a good forecast?
2. **Predictor Variables:** Which weather variables matter? What non-weather factors are important?
3. **Model Plan:** What data do you need? How will you split train/test? What metrics to evaluate?
4. **Anticipated Challenges:** What factors are hard to predict? How to handle special events?


<div class="alert alert-block alert-success">
<b>Your turn:</b><br>
Complete the forecasting plan template below. Write directly in the code cell to produce portfolio evidence.

<details>
    <summary style="color:black;font-weight:bold">Click here for a hint</summary>

**Problem definition hints:**
- Dependent variable: Daily visitor count
- Success criteria: ±20 visitors for a typical 100-visitor day (±20%)
- Operational use: Schedule right number of staff (1 per 30 visitors)

**Predictor variables hints:**
- Weather: Temperature, precipitation, wind speed, cloud cover
- Calendar: Day of week (weekends vs weekdays), school holidays, public holidays
- Events: Scheduled programmes, author visits, school group bookings

**Challenge hints:**
- Unexpected closures (system failures, maintenance)
- Competing attractions on the same day
- Minimum staffing regardless of forecast (cannot staff below 2 librarians)
</details>

<details>
    <summary style="color:green;font-weight:bold">Click here for the solution</summary>

```python
forecasting_plan = {
    'business_problem': {
        'prediction_target': 'Daily library visitor count',
        'success_criteria': 'Forecast within ±20 visitors (±20% of typical 100-visitor day)',
        'operational_use': [
            'Staff scheduling: 1 librarian per 30 visitors expected',
            'Resource allocation: Computer availability, returns volumes',
            'Programme planning: Scale activities to expected attendance'
        ]
    },
    'predictor_variables': {
        'weather': ['temperature_2m', 'precipitation', 'wind_speed', 'cloud_cover'],
        'calendar': ['day_of_week', 'is_weekend', 'school_holiday', 'public_holiday'],
        'events': ['scheduled_programme', 'author_visit', 'school_group_booking'],
        'strongest_predictors': ['day_of_week', 'school_holiday', 'precipitation']
    },
    'model_plan': {
        'data_needed': [
            'Historical visitor counts (minimum 2 years)',
            'Weather data for matching dates',
            'Council calendar (holidays, events)',
            'Closure records (to exclude from training)'
        ],
        'train_test_split': '80% training (older data), 20% testing (most recent data)',
        'evaluation_metrics': ['R² — overall fit', 'RMSE — forecast error in visitors', 'MAE — average daily error']
    },
    'challenges': {
        'difficult_factors': [
            'Unexpected closures (system failures, water damage)',
            'Competing attractions (nearby festival, school events)',
            'Viral social media trends driving sudden increased interest'
        ],
        'special_events': 'Flag scheduled events as binary features; consider separate models for event vs non-event days',
        'operational_constraints': 'Minimum 2 staff always required regardless of forecast; primary saving is avoiding overtime'
    }
}
```
</details>
</div>


In [ ]:
# Your forecasting plan (complete for portfolio evidence)
forecasting_plan = {
    'business_problem': {
        'prediction_target': '',
        'success_criteria': '',
        'operational_use': []
    },
    'predictor_variables': {
        'weather': [],
        'calendar': [],
        'events': [],
        'strongest_predictors': []
    },
    'model_plan': {
        'data_needed': [],
        'train_test_split': '',
        'evaluation_metrics': []
    },
    'challenges': {
        'difficult_factors': [],
        'special_events': '',
        'operational_constraints': ''
    }
}

# Display your plan
print("=== LIBRARY VISITOR FORECASTING PLAN ===")
for section, details in forecasting_plan.items():
    print(f"\n{section.upper().replace('_', ' ')}:")
    if isinstance(details, dict):
        for k, v in details.items():
            print(f"  {k}: {v}")
    else:
        print(f"  {details}")

**Portfolio note:** This forecasting plan demonstrates your ability to select and apply appropriate data-gathering and analysis methods (MLO 3). Include it in your Module Challenge Part A portfolio. Your plan should clearly identify the business problem, justify your choice of predictor variables, and acknowledge limitations.


---

## Prototyping Tools

The Python workflow above is powerful but not always the right starting point. Before committing to a full Python implementation, you can use prototyping tools to quickly explore different approaches and demonstrate concepts to non-technical stakeholders.

### Microsoft Excel (Analysis ToolPak)

Excel's built-in Analysis ToolPak provides regression and statistical functions accessible through a familiar interface.

- **When to use:** Small datasets (<10,000 rows), stakeholders already familiar with Excel
- **Advantages:** Accessible, easy to share, built-in statistical functions
- **Limitations:** Limited scalability; manual updates required for new data
- **Documentation:** [https://support.microsoft.com/excel](https://support.microsoft.com/excel) — search "Analysis ToolPak"

### Orange Data Mining

Orange uses the same Python packages (scikit-learn, statsmodels) as this notebook but provides a visual drag-and-drop interface for building and comparing models.

- **When to use:** Visual workflow design, testing multiple models quickly
- **Advantages:** No coding required, visual model comparison, interactive exploration
- **Limitations:** Limited data import options; not suitable for production deployment, requires additional setup
- **Link:** [https://orangedatamining.com/](https://orangedatamining.com/)

### JASP

JASP is statistics-focused and draws on powerful R packages. It is well-suited to hypothesis testing and academic analysis.

- **When to use:** Fast statistical analysis or model forcasting requiring academic rigour or advanced but well established statistical tools
- **Advantages:** Publication-ready outputs, extensive statistical tests, easily installable from the Windows Store or their website
- **Limitations:** Statistics-focused rather than deployment-oriented. Backend uses R rather than Python, so not directly transferable to production code.
- **Link:** [https://jasp-stats.org/](https://jasp-stats.org/)

**Business Context:** Use prototyping tools to validate your modelling approach and communicate results to non-technical stakeholders before investing time in production-quality Python code.


---

## Reflection

Reflect on the following two questions and record your thoughts in the cell below. Ground your answers in examples from your workplace, using **approximately 300 words** in total.

Your answers here form part of your Module Challenge Part A portfolio and are assessed against **MLO3:** *Select and apply appropriate data-gathering methods for a given data analysis scenario*.

Where you draw on external sources, cite them in Harvard style. This does not count towards your word count.

Once you commit your notebook, your Coach will be able to provide feedback on your responses. You can develop your answers further before your final submission.

<div class="alert alert-block alert-info">

1. We removed sunshine duration from Model 1 to create simpler Model 2 with only a tiny performance loss. In your workplace, when would you prioritise model simplicity over maximum accuracy? What stakeholder factors influence this decision?

2. The test-train split confirmed our model generalises well, with only a small performance drop. Describe a workplace scenario where a model might perform well on historical data but fail on future data. What changed?

</div>

*Write your responses here.*

1. 

2. 

References (if applicable):

---

## Summary

**You have completed Week 7 Section 1.** You can now:

- Defined a business problem and success criteria for forecasting outdoor event attendance from weather conditions
- Explored the Open-Meteo weather dataset using descriptive statistics and a correlation matrix, identifying multicollinearity between temperature and sunshine duration
- Built a linear regression model using all five weather variables and evaluated it with R², RMSE, and MAE
- Built a simplified model by removing sunshine_duration to address multicollinearity, and compared its performance against Model 1
- Validated the preferred model using an 80/20 train-test split, confirming it generalises to unseen data with minimal overfitting
- Designed a complete forecasting plan for a library visitor scenario, applying the same analytical workflow independently

**Key takeaways:**

- If two predictor variables correlate more strongly with each other than either does with the dependent variable, consider removing one to simplify the model without materially reducing performance.
- R² measures the proportion of variance explained, while RMSE and MAE express forecast error in the same units as the target variable, making them easier to communicate to stakeholders.

**What comes next:**

- **Section 2: Analysing Data Through Time** extends this validation approach to data that changes over time, using decomposition and Holt-Winters exponential smoothing to forecast future values without external predictor variables.

**Action items:**

1. Save this notebook (Ctrl+S or Cmd+S)
2. Commit and push your work to your GitHub repository with the commit message `"Complete Week 7 Section 1 Working Through a Data Analysis"`
3. Make sure the repository shows your latest commit and that all files are up to date
4. Share a link to your repository with your Coach in the DT401.7 component on Aptem
5. Open `DT401_Week_7_Section_2.ipynb` when ready to continue